# Hipótesis de Viralidad

**Objetivo:** Predecir si un artista tiene >70% de probabilidad de duplicar sus 'Deezer Fans' en 6 meses.

**Features (Variables predictoras):**
1. Deezer Rank actual.
2. Ratio de crecimiento de Fans (simulado por ahora).
3. Popularidad del Top Track vs. Promedio del género.

**Target (Variable a predecir):** `is_viral` (1 = Sí, 0 = No).

## Product Thinking — por qué esto importa

El dashboard responde *"¿cómo están los artistas hoy?"*. Este modelo responde *"¿quién explotará en 6 meses?"* — la pregunta que permite firmar talento **antes** de que su precio suba.

**Datos actuales:** 10 artistas reales (snapshot Deezer) — suficiente para diseñar features, NO para entrenar. La siguiente lección genera un dataset sintético de 10.000 artistas con series de tiempo; este notebook queda preparado para consumir cualquiera de las dos fuentes.

In [ ]:
# Entorno del laboratorio de ML
import sys
import sklearn
import numpy as np
import pandas as pd
import matplotlib
import seaborn as sns

print(f"Python {sys.version.split()[0]}")
print(f"scikit-learn {sklearn.__version__} | pandas {pd.__version__} | numpy {np.__version__}")
print(f"matplotlib {matplotlib.__version__} | seaborn {sns.__version__}")

In [ ]:
# Datos reales del warehouse (con auto-reparación si no existe)
import os
sys.path.insert(0, os.path.abspath(".."))

import duckdb
import bootstrap_db

DB_PATH = bootstrap_db.resolve_db_path()
bootstrap_db.ensure_database(DB_PATH)
bootstrap_db.ensure_table(DB_PATH)

con = duckdb.connect(DB_PATH, read_only=True)
df_real = pd.read_sql_query("SELECT * FROM artist_scouting_deezer", con)
con.close()

print(f"Artistas reales cargados: {len(df_real)}")
print(f"Warehouse: {DB_PATH}")
df_real[["artist_name", "deezer_fans", "deezer_rank", "top_track_rank", "scouting_score", "ar_recommendation"]]

## Features disponibles hoy (de los datos reales)

| Feature | Fuente | Rol en la hipótesis |
|---------|--------|---------------------|
| `deezer_rank` | API real (proxy top-10) | Feature 1: posición actual |
| `deezer_fans` | API real | Base para el ratio de crecimiento (Feature 2, con histórico) |
| `top_track_rank` | API real | Proxy de popularidad del hit (Feature 3) |
| `fan_rank_ratio` | mart dbt | Eficiencia de conversión oyente→fan |

## Pendiente (próxima lección)
- Dataset sintético de 10.000 artistas con variables de tiempo.
- Construcción del target `is_viral` (duplicación de fans en 6 meses).
- Baseline: Regresión Logística → luego Random Forest / XGBoost.